In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
from litellm import completion
load_dotenv(override=True)
MODEL = 'gemini/gemini-3-flash-preview'

In [4]:
deals = ScrapedDeal.fetch(show_progress=True)

 33%|█████████████████████████████████▋                                                                   | 1/3 [00:55<01:50, 55.18s/it]D:\Rohan\Machine Learning and AI\LLM-apps\PriceIsRight\agents\deals.py:29: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  description = BeautifulSoup(description, "html.parser").get_text()
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [02:55<00:00, 58.51s/it]


In [5]:
len(deals)

30

In [6]:
deals[10].describe()

'Title: Open-Box Microsoft Surface Laptop 3 Intel i7 13.5" Laptop for $310 + free shipping\nDetails: You can get this Microsoft Surface laptop for just $310 today at eBay. You\'d still pay the full $700 for this model today at Amazon. The model at eBay is an open-box laptop, which means it\'s new, but won\'t ship in its original packaging. Buy Now at eBay\nFeatures: Intel i7-1065G7 4-core processor 13.5" 2256 x 1504 display 16GB RAM, 256GB SSD Model: VEF-00022\nURL: https://www.dealnews.com/products/Microsoft/Microsoft-Surface-Laptop-3-Intel-i7-13-5-Laptop/497253.html?iref=rss-c39'

### lets ask llms to summarize the deals and identify their prices

In [2]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [3]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [4]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

NameError: name 'deals' is not defined

In [20]:
response = completion(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.content
results

'{\n  "deals": [\n    {\n      "product_description": "This high-performance laptop features a 13th Generation Intel Core i5-1334U 10-core processor and a 15.6-inch Full HD touchscreen display. It is equipped with 8GB of RAM and a 512GB solid-state drive for efficient multitasking and ample storage. The system runs Windows 11 Home in S Mode and comes in a sleek black finish.",\n      "price": 350,\n      "url": "https://www.dealnews.com/products/Dell/Dell-13-th-Gen-i5-15-6-2-K-Touchscreen-Laptop-w-512-GB-SSD/495515.html?iref=rss-c39"\n    },\n    {\n      "product_description": "These premium wireless earbuds offer high-quality audio through 11mm woofers and 6.5mm tweeters. They feature intelligent active noise cancellation to block out environmental sounds and are built with water-resistant materials for durability during workouts. The set includes a matching charging case and provides a secure, ergonomic fit.",\n      "price": 29,\n      "url": "https://www.dealnews.com/products/Sams

In [21]:
# because i am using gemini modell and litellm, it does not support direct parsing like openai. So need to implement below method 
import json
from types import SimpleNamespace

results = json.loads(results, object_hook=lambda d: SimpleNamespace(**d))
results

In [22]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


This high-performance laptop features a 13th Generation Intel Core i5-1334U 10-core processor and a 15.6-inch Full HD touchscreen display. It is equipped with 8GB of RAM and a 512GB solid-state drive for efficient multitasking and ample storage. The system runs Windows 11 Home in S Mode and comes in a sleek black finish.
350
https://www.dealnews.com/products/Dell/Dell-13-th-Gen-i5-15-6-2-K-Touchscreen-Laptop-w-512-GB-SSD/495515.html?iref=rss-c39

These premium wireless earbuds offer high-quality audio through 11mm woofers and 6.5mm tweeters. They feature intelligent active noise cancellation to block out environmental sounds and are built with water-resistant materials for durability during workouts. The set includes a matching charging case and provides a secure, ergonomic fit.
29
https://www.dealnews.com/products/Samsung/Samsung-Galaxy-Buds-Pro-Wireless-Headphones/249854.html?iref=rss-c142

This advanced robot vacuum features LiDAR navigation for precise mapping and adjustable suctio

In [5]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [6]:
from agents.scanner_agent import ScannerAgent

In [7]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
D:\Rohan\Machine Learning and AI\LLM-apps\PriceIsRight\agents\deals.py:29: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  description = BeautifulSoup(description, "html.parser").get_text()
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling Gemini using Structured Outputs
23:46:40 - LiteLLM:INFO: utils.py:3874 - 
LiteLLM completion() model= gemini-3-flash-preview; provider = gemini
INFO:LiteLLM:
LiteLLM completion() model= gemini-3-flash-preview; provider = gemini
23:46:47 - LiteLLM:INFO: utils.py:1623 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:ro

In [8]:
result

namespace(deals=[namespace(product_description='This high-performance laptop features the Intel Core Ultra 7 256V 8-core processor and a 14-inch 1920x1200 non-touch display. It is equipped with 16GB of RAM and a substantial 1TB SSD for extensive storage and multitasking capabilities. The system comes pre-installed with Windows 11 Home, making it ready for modern computing needs.',
                           price=530,
                           url='https://www.dealnews.com/products/Acer/Acer-Aspire-14-Core-Ultra-7-14-Laptop/497644.html?iref=rss-c39'),
                 namespace(product_description='This 34-inch curved gaming monitor offers an immersive ultrawide 3440x1440p resolution and a smooth 165Hz refresh rate. It supports AMD FreeSync technology to reduce screen tearing and stuttering during intensive gameplay. Connectivity options include both DisplayPort and HDMI inputs for versatile setup configurations.',
                           price=197,
                           url='

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [9]:
load_dotenv(override=True)

True

In [10]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [11]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [12]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [13]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [14]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [15]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message
00:11:03 - LiteLLM:INFO: utils.py:3874 - 
LiteLLM completion() model= gemini-3-flash-preview; provider = gemini
INFO:LiteLLM:
LiteLLM completion() model= gemini-3-flash-preview; provider = gemini
00:11:04 - LiteLLM:INFO: utils.py:1623 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
